# **===========================================================================================================================================================** #
# **Course End Project SandBox Notebook** #
# **===========================================================================================================================================================** #
**Sandbox Notebook**: For running quick tests and exploration.</br>


## Setup

### Imports

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'    # must be first

import time
import subprocess
#import tensorflow as tf                     
#import tf_keras
#import transformers

# from transformers.utils import is_tf_available
# print(f'TF available to transformers: {is_tf_available()}')
# from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification

import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import requests
import json

from itertools import product
from collections import Counter


#from tensorflow.keras.preprocessing.text import Tokenizer
#from tensorflow.keras.preprocessing.sequence import pad_sequences
# from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
# from tensorflow.keras.optimizers import Adam
#from tensorflow.keras.metrics import Precision, Recall, AUC

# from sklearn.metrics import (
#     accuracy_score,                         
#     confusion_matrix, roc_curve, auc,
#     precision_recall_curve, ConfusionMatrixDisplay,
#     classification_report, precision_score,
#     recall_score, f1_score, roc_auc_score,
#     balanced_accuracy_score
# )

# from sklearn.naive_bayes import MultinomialNB
# from sklearn.model_selection import train_test_split
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.preprocessing import label_binarize
# from sklearn.svm import LinearSVC
# from sklearn.calibration import CalibratedClassifierCV
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import GridSearchCV
# from sklearn.preprocessing import LabelEncoder

# Initialize the vectorizer
# tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

# from nltk.tokenize import word_tokenize, sent_tokenize
# from nltk.corpus import stopwords
# from nltk.stem import PorterStemmer, WordNetLemmatizer
# from nltk.sentiment.vader import SentimentIntensityAnalyzer

# from wordcloud import WordCloud
# from bs4 import BeautifulSoup
# from spellchecker import SpellChecker

import warnings
# from sklearn.exceptions import ConvergenceWarning
# warnings.filterwarnings('ignore', category=FutureWarning)
# warnings.filterwarnings('ignore', category=ConvergenceWarning)

# result = subprocess.run(['pip', 'show', 'tf-keras'], capture_output=True, text=True)
# print(result.stdout)


#### Track Notebook Runtime Performance ####

In [ ]:
# For Tracking Notebook runtime/performance
notebook_start_time = time.time()

### Utility Functions
These utility/support methods will eventually be pushed into a private support library (Yes, I know, I know....).

#### EDA Utility Functions ####

In [ ]:
target_label = ''

def WrapText(text, max_width=15):
    """Wrap text to multiple lines"""
    words = str(text).split()
    lines = []
    current_line = ""
    
    for word in words:
        if len(current_line + " " + word) <= max_width:
            current_line = (current_line + " " + word).strip()
        else:
            if current_line:
                lines.append(current_line)
            current_line = word
    if current_line:
        lines.append(current_line)
    
    return "\n".join(lines)

def DisplayTable(df_target, table_title=None, max_cell_length=30, show_index=False, 
                 wrap_headers=True, header_wrap_width=15, min_height=3,
                 row_height=0.35, font_size=10):

    if df_target.empty:
        print(f"No data to display{': ' + table_title if table_title else ''}")
        return

    n_rows, n_cols = df_target.shape
    
    # Adjust columns if showing index
    if show_index:
        n_cols += 1

    # Calculate width based on longest column name or cell content
    col_widths = []
    for col in df_target.columns:
        if wrap_headers:
            max_len = max(len(line) for line in WrapText(col, header_wrap_width).split('\n'))
        else:
            max_len = len(str(col))
        for val in df_target[col]:
            val_len = len(f"{val:.2f}" if isinstance(val, float) else str(val))
            max_len = max(max_len, val_len)
        col_widths.append(min(max_len, max_cell_length))
    
    # Dynamic figure width based on content
    fig_width = max(sum(col_widths) * 0.15, n_cols * 2.0)
    
    # Calculate extra height for wrapped headers
    if wrap_headers:
        max_header_lines = max(len(WrapText(col, header_wrap_width).split('\n')) for col in df_target.columns)
    else:
        max_header_lines = 1
    
    fig_height = (n_rows + max_header_lines) * row_height
    if table_title:
        fig_height += 0.4
    
    # Ensure minimum height for small tables
    fig_height = max(fig_height, min_height)
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    # Format floats and truncate long text
    cell_text = []
    for idx, row in zip(df_target.index, df_target.values):
        new_row = []
        
        # Add index as first column if show_index
        if show_index:
            s = str(idx)
            if len(s) > max_cell_length:
                s = s[:max_cell_length - 3] + '...'
            new_row.append(s)
        
        for val in row:
            if isinstance(val, (float)) and not isinstance(val, bool):
                new_row.append(f"{val:.2f}")
            else:
                s = str(val)
                if len(s) > max_cell_length:
                    s = s[:max_cell_length - 3] + '...'
                new_row.append(s)
        cell_text.append(new_row)

    # Build column labels (with wrapping)
    if show_index:
        col_labels = [df_target.index.name or '']
        if wrap_headers:
            col_labels += [WrapText(col, header_wrap_width) for col in df_target.columns]
        else:
            col_labels += list(df_target.columns)
    else:
        if wrap_headers:
            col_labels = [WrapText(col, header_wrap_width) for col in df_target.columns]
        else:
            col_labels = list(df_target.columns)

    table = ax.table(
        cellText=cell_text,
        colLabels=col_labels,
        cellLoc='center',
        loc='upper center',
        bbox=[0, 0, 1, 1]
    )

    # Bold column headers and set header background
    for col in range(n_cols):
        table[(0, col)].set_facecolor('#4a90d9')
        table[(0, col)].set_text_props(color='white', fontweight='bold')

    # Style data cells
    for row in range(1, n_rows + 1):
        for col in range(n_cols):
            if show_index and col == 0:
                table[(row, col)].set_facecolor('#b8d4e8')
                table[(row, col)].set_text_props(fontweight='bold')
            else:
                table[(row, col)].set_facecolor('#d4e6f1')

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.auto_set_column_width(col=list(range(n_cols)))

    if table_title is not None:
        fig.suptitle(table_title, fontweight='bold', fontsize=14)

    plt.tight_layout(rect=[0, 0, 1, 0.95] if table_title else [0, 0, 1, 1])

    plt.show()
    plt.close(fig)
    print()
    print()
    
def PrintDataFrameStatistics(dataset_name, df_target, display_table=True):
    # Print Stats
    print("***********************************")
    print("Description Stats")
    print("***********************************")
    print()

    # Capture for Table plot
    df_stats = df_target.describe(include='all').T.reset_index()
    df_stats.rename(columns={'index': 'Feature'}, inplace=True)
    print(df_stats)
    print()

    # Print df Column Info
    print("***********************************")
    print("Basic Info of imported data set")
    print("***********************************")
    print()

    df_info = pd.DataFrame({
                            'Feature': df_target.columns,
                            'Non-Null Count': df_target.notna().sum().values,
                            'Null Count': df_target.isna().sum().values,
                            'Dtype': df_target.dtypes.values
                            }).reset_index(drop=True)

    print(f'Dataset Shape:{df_target.shape}')
    df_shape = pd.DataFrame({'Rows': [df_target.shape[0]], 'Columns': [df_target.shape[1]]})
    print()

    print('Do we have any features with null values?:')
    print(df_target.isnull().any().any())
    print()

    print('Do we have any features empty strings?:')
    print((df_target == "").any())
    print()

    print('Feature Columns with that have null values:')
    print(df_target.isnull().sum()[df_target.isnull().sum() > 0])

    # Capture for Table plot
    cols_to_plot = df_target.select_dtypes(exclude=['number']).columns
    missing = df_target[cols_to_plot].isnull().sum()
    missing = missing[missing > 0]
    df_missing = pd.DataFrame({
                                'Feature': missing.index,
                                'Missing Count': missing.values,
                                'Missing %': (missing.values / len(df_target) * 100).round(2)
                                }).reset_index(drop=True)
    print()

    print('Do we have any features with nan values?:')

    cols_to_plot = df_target.select_dtypes(include=['number']).columns
    nan_vals = df_target[cols_to_plot].isna().sum()
    nan_vals = nan_vals[nan_vals > 0]
    # Capture for Table plot
    df_nan = pd.DataFrame({
                            'Feature': nan_vals.index,
                            'NaN Count': nan_vals.values,
                            'NaN %': (nan_vals.values / len(df_target) * 100).round(2)
                        }).reset_index(drop=True)
    print(df_target.isna().any().any())

    # Sum up the number of missing features per row
    missing_per_row = df_target.isna().sum(axis=1)  # count missing per row
    missing_counts = missing_per_row.value_counts().sort_index()  # count rows for each missing count

    df_missingfeature_rowcounts = pd.DataFrame({
        'Missing Features': missing_counts.index,
        'Row Count': missing_counts.values
    })

    sample_size = min(20, len(df_target))

    print("***********************************")
    print("First N rows of Data")
    print("***********************************")
    print()
    print(df_target.head(sample_size))
    print()

    print("***********************************")
    print("First N rows of Random Sample Data")
    print("***********************************")
    print()
    df_randomsample = df_target.sample(n=sample_size)
    print(df_target.sample(sample_size))
    print()

    #
    # Display Results in Pretty Tables
    #
    if display_table == True:
        print('Display Analysis Results in Tables')
        DisplayTable(df_stats, f'{dataset_name} Description Statistics')
        print()
        DisplayTable(df_info, f'{dataset_name} Basic Information')
        print()
        DisplayTable(df_shape, f'{dataset_name} Dataset Shape')
        print()
        DisplayTable(df_missing, f'{dataset_name} Missing Categorical (String) Data')
        print()
        DisplayTable(df_nan, f'{dataset_name} Missing Numeric Data')
        print()
        DisplayTable(df_missingfeature_rowcounts, f'{dataset_name} Summary of Missing Feature Row Counts')
        print()    
        DisplayTable(df_randomsample, f'{dataset_name} Random Data Sample')
        
    print()

#### Plotting Utility Functions ####

In [ ]:
def DisplayBarPlot(df_Target, x_axis, y_axis, 
                    x_axis_label=None,
                    y_axis_label='Count',
                    plot_title=None, 
                    colors=None,
                    fmt='%.0f'):

    # Plot
    plt.figure(figsize=(12, 6))

    if colors:
        ax = sns.barplot(x=x_axis, 
                        y=y_axis, 
                        data=df_Target, 
                        hue=x_axis, 
                        palette=colors,
                        legend=False)
    else:
        ax = sns.barplot(x=x_axis, 
                        y=y_axis, 
                        data=df_Target, 
                        hue=x_axis, 
                        palette='Blues_d', 
                        legend=False)

    # Bold title and axis labels
    plt.title(plot_title, fontweight='bold')
    plt.xlabel(x_axis_label, fontweight='bold')
    plt.ylabel(y_axis_label, fontweight='bold')

    # Bold tick labels
    plt.xticks(rotation=45, ha='right', fontweight='bold')
    plt.yticks(fontweight='bold')

    # Add white count labels inside bars
    for container in ax.containers:
        ax.bar_label(container, color='white', fontweight='bold', label_type='center', fmt=fmt)

    plt.tight_layout()
    plt.show()

def DisplayCountPlot(df_Target, x_axis, order, plot_title, fig_size, hue_value=None, plot_labels=None, displayLabels=True):

    if hue_value is not None:
        hue = hue_value
    else:
        hue = x_axis

    plt.figure(figsize=fig_size)
    ax = sns.countplot(x=x_axis, 
                       data=df_Target, 
                       order=order, 
                       hue=hue, 
                       palette='Blues_d', 
                       legend=False)

    plt.title(plot_title, fontweight='bold')
    plt.xlabel(x_axis, fontweight='bold')
    plt.ylabel('Count', fontweight='bold')
    plt.xticks(rotation=90, fontweight='bold')
    plt.yticks(fontweight='bold')

    # Skip bar labels for Year (too many bars)
    if displayLabels == True:
        for container in ax.containers:
            ax.bar_label(container, color='white', fontweight='bold', label_type='center')

    if plot_labels is not None:
        plt.legend(title=hue, labels=plot_labels)

    plt.tight_layout()
    plt.show()

def DisplayGroupedCountPlot(df_target, x_axis, hue_col, plot_title, fig_size, top_n=None):
    
    # Optionally limit to top N categories on x_axis
    if top_n is not None:
        top_cats = df_target[x_axis].value_counts().head(top_n).index
        df_target = df_target[df_target[x_axis].isin(top_cats)]

    plt.figure(figsize=fig_size)
    ax = sns.countplot(x=x_axis,
                       data=df_target,
                       hue=hue_col,
                       palette='Blues_d',
                       order=df_target[x_axis].value_counts().index)

    plt.title(plot_title, fontweight='bold')
    plt.xlabel(x_axis, fontweight='bold')
    plt.ylabel('Count', fontweight='bold')
    plt.xticks(rotation=90, fontweight='bold')
    plt.yticks(fontweight='bold')
    plt.legend(title=hue_col, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()   

def PlotLearningCurves(history, plot_title):

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    axes[0].set_title(f'{plot_title}: loss')
    axes[0].plot(history['loss'], label='Training')
    axes[0].plot(history['val_loss'], label='Validation')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (binary crossentropy)')
    axes[0].legend(loc='best')

    axes[1].set_title(f'{plot_title}: accuracy')
    axes[1].plot(history['accuracy'], label='Training')
    axes[1].plot(history['val_accuracy'], label='Validation')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend(loc='best')

    plt.tight_layout()
    plt.show()

def PlotConfusionMatrix(model, X_test, y_test, plot_title):

    y_probs = model.predict(X_test).flatten()
    y_pred = (y_probs >= 0.5).astype(int)

    print('----------------------------------------------------------')
    print(plot_title)
    print()
    print(classification_report(y_test, y_pred, target_names=['Not Recommend', 'Recommend']))
    print(f'ROC-AUC: {roc_auc_score(y_test, y_probs):.4f}')
    print('----------------------------------------------------------')

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Recommend', 'Recommend'])
    disp.plot(cmap='Blues')
    plt.title(plot_title)
    plt.show() 


#### Model Comparison Utility Functions ####

In [ ]:
def GetPredictions(name, results):
    model  = results['model']
    X_test = results['X_test']

    if name == 'DistilBERT':
        logits = model.predict(
                     {'input_ids':      X_test['input_ids'],
                      'attention_mask': X_test['attention_mask']}
                 ).logits
        probs  = tf.nn.softmax(logits, axis=-1).numpy()
        return np.argmax(probs, axis=-1)
    else:
        return model.predict(X_test)


def PlotConfusionMatrices(results_dict):
    n_models = len(results_dict)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))

    if n_models == 1:
        axes = [axes]

    palettes = ['Blues', 'Greens', 'Oranges', 'Purples']

    for i, (name, results) in enumerate(results_dict.items()):
        y_test           = results['y_test']
        test_predictions = GetPredictions(name, results)
        cm               = confusion_matrix(y_test, test_predictions)

        sns.heatmap(cm, annot=True, fmt='d', cmap=palettes[i % len(palettes)],
                    ax=axes[i], cbar=True)
        axes[i].set_title(f'{name}', fontweight='bold')
        axes[i].set_xlabel('Predicted')
        axes[i].set_ylabel('Actual')

    plt.suptitle('Confusion Matrix Comparison', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    print()


def CompareClassificationResults(results_dict, title='Classifier Comparison'):

    sns.set_style("whitegrid")

    #
    # 1. Calculate all metrics
    #
    metrics_data = []
    for name, results in results_dict.items():
        y_test           = results['y_test']
        test_predictions = GetPredictions(name, results)
        report           = results['report']

        metrics_data.append({
            'Model':             name,
            'Train Accuracy':    results['train_accuracy'],
            'Test Accuracy':     results['test_accuracy'],
            'Balanced Accuracy': balanced_accuracy_score(y_test, test_predictions),
            'Precision':         precision_score(y_test, test_predictions, average='weighted'),
            'Recall':            recall_score(y_test,    test_predictions, average='weighted'),
            'F1 Score':          results['f1_score'],
            'Gap':               results['train_accuracy'] - results['test_accuracy']
        })

    df_comparisons = pd.DataFrame(metrics_data).set_index('Model')

    #
    # 2. Print Metrics Table
    #
    print('=' * 70)
    print(title)
    print('=' * 70)
    print(df_comparisons.round(4))
    print()

    DisplayTable(df_comparisons, 'Classification Comparisons', show_index=True)

    #
    # 3. Accuracy Comparison (Train vs Test vs Balanced)
    #
    accuracy_metrics = []
    for name, results in results_dict.items():
        y_test           = results['y_test']
        test_predictions = GetPredictions(name, results)

        accuracy_metrics.append({'Model': name, 'Metric': 'Train Accuracy',    'Score': results['train_accuracy']})
        accuracy_metrics.append({'Model': name, 'Metric': 'Test Accuracy',     'Score': results['test_accuracy']})
        accuracy_metrics.append({'Model': name, 'Metric': 'Balanced Accuracy', 'Score': balanced_accuracy_score(y_test, test_predictions)})

    acc_df = pd.DataFrame(accuracy_metrics)

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(data=acc_df, x='Metric', y='Score', hue='Model', palette='deep')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=9)
    plt.title('Accuracy Metrics Comparison', fontweight='bold')
    plt.ylim(0, 1.1)
    plt.legend(title='Model')
    plt.tight_layout()
    plt.show()
    print()

    #
    # 4. Precision / Recall / F1 Comparison
    #
    prf_metrics = []
    for name, results in results_dict.items():
        y_test           = results['y_test']
        test_predictions = GetPredictions(name, results)

        prf_metrics.append({'Model': name, 'Metric': 'Precision', 'Score': precision_score(y_test, test_predictions, average='weighted')})
        prf_metrics.append({'Model': name, 'Metric': 'Recall',    'Score': recall_score(y_test,    test_predictions, average='weighted')})
        prf_metrics.append({'Model': name, 'Metric': 'F1 Score',  'Score': results['f1_score']})

    prf_df = pd.DataFrame(prf_metrics)

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(data=prf_df, x='Metric', y='Score', hue='Model', palette='Set2')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=9)
    plt.title('Precision / Recall / F1 Comparison', fontweight='bold')
    plt.ylim(0, 1.1)
    plt.legend(title='Model')
    plt.tight_layout()
    plt.show()
    print()

    #
    # 5. Gap Comparison (Overfitting indicator)
    #
    gap_data = pd.DataFrame({
        'Model': list(results_dict.keys()),
        'Gap':   [r['train_accuracy'] - r['test_accuracy'] for r in results_dict.values()]
    })

    plt.figure(figsize=(10, 5))
    ax = sns.barplot(data=gap_data, x='Model', y='Gap', palette='deep')
    ax.bar_label(ax.containers[0], fmt='%.4f', fontsize=10)
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    plt.title('Overfitting Indicator (Lower is Better)', fontweight='bold')
    plt.ylabel('Train - Test Accuracy Gap')
    plt.tight_layout()
    plt.show()
    print()

    #
    # 6. Confusion Matrices
    #
    PlotConfusionMatrices(results_dict)

    #
    # 7. Metrics Heatmap
    #
    plt.figure(figsize=(12, len(results_dict) * 0.8 + 2))
    heatmap_cols = ['Test Accuracy', 'Balanced Accuracy', 'Precision', 'Recall', 'F1 Score']
    sns.heatmap(df_comparisons[heatmap_cols], annot=True, fmt='.4f', cmap='YlGnBu',
                vmin=0, vmax=1, linewidths=0.5)
    plt.title('Model Metrics Heatmap', fontweight='bold')
    plt.tight_layout()
    plt.show()

    #
    # 8. Plot Classification Report Results
    #
    for i, (name, results) in enumerate(results_dict.items()):
        report    = results['report']
        df_result = pd.DataFrame(report)
        DisplayTable(df_result,
                     f'{name} Classification Report',
                     show_index=True)

    n_models = len(results_dict)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 4))

    if n_models == 1:
        axes = [axes]

    for i, (name, results) in enumerate(results_dict.items()):
        report    = results['report']
        df_report = pd.DataFrame(report).T
        df_report = df_report.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
        df_report = df_report.drop(columns=['support'], errors='ignore')

        sns.heatmap(df_report, annot=True, fmt='.2f', cmap='RdYlGn',
                    vmin=0, vmax=1, ax=axes[i], linewidths=0.5)
        axes[i].set_title(f'{name}', fontweight='bold')
        axes[i].set_ylabel('Class')
        axes[i].set_xlabel('Metric')

    plt.suptitle('Classification Reports Comparison', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    #
    # 9. ROC Curves
    #
    plt.figure(figsize=(10, 6))
    for name, results in results_dict.items():
        if 'y_test_prob' not in results:
            continue
        y_test      = results['y_test']
        y_test_prob = results['y_test_prob']
        classes     = sorted(np.unique(y_test))
        roc_auc     = roc_auc_score(y_test, y_test_prob, multi_class='ovr', average='weighted')
        fpr, tpr, _ = roc_curve(label_binarize(y_test, classes=classes).ravel(),
                                y_test_prob.ravel())
        plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate', fontweight='bold')
    plt.ylabel('True Positive Rate',  fontweight='bold')
    plt.title('ROC Curves Comparison', fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.show()
    print()

    #
    # 10. Precision-Recall Curves
    #
    plt.figure(figsize=(10, 6))
    for name, results in results_dict.items():
        if 'y_test_prob' not in results:
            continue
        y_test      = results['y_test']
        y_test_prob = results['y_test_prob']
        classes     = sorted(np.unique(y_test))
        precision, recall, _ = precision_recall_curve(
            label_binarize(y_test, classes=classes).ravel(),
            y_test_prob.ravel()
        )
        pr_auc = auc(recall, precision)
        plt.plot(recall, precision, label=f'{name} (AUC = {pr_auc:.3f})')

    plt.xlabel('Recall',    fontweight='bold')
    plt.ylabel('Precision', fontweight='bold')
    plt.title('Precision-Recall Curves Comparison', fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.show()
    print()

    return df_comparisons

## 1.0 Load Data ## 

In [ ]:
# df_complaints = pd.read_csv('../../data/Unit5EndProject/complaints_banking_2023.csv')
# df_departments = pd.read_csv('../../data/Unit5EndProject/Department_of_Product.csv')
# df_issues = pd.read_csv('../../data/Unit5EndProject/Issues.csv')

# # Make copies of the original unmodified data
# df_complaints_Original = df_complaints.copy(deep=True)
# df_departments_Original = df_departments.copy(deep=True)
# df_issues_Original = df_issues.copy(deep=True)

API_KEY = 'd6f1489c0e044b0a95ea2e04eb8ff3a6'
TOTAL_WANTED = 150
RESULTS_PER_CALL = 100
all_ingredients = []

for offset in range(0, TOTAL_WANTED, RESULTS_PER_CALL):
    url = (
                f"https://api.spoonacular.com/food/ingredients/search"
                f"?query=a"                        # ✅ required — use a broad term or wildcard
                f"&number={RESULTS_PER_CALL}"
                f"&offset={offset}"
                f"&apiKey={API_KEY}"
            )
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        all_ingredients.extend(data['results'])
    else:
        print(f"Error at offset {offset}: {response.status_code}")
        print(response.json())   # ← shows the exact error message from Spoonacular
        break

# Save locally to avoid repeated API calls during development
with open('pantry_ingredients_2000.json', 'w') as f:
    json.dump(all_ingredients, f)


# Load into panda dataframe
with open('pantry_ingredients_2000.json', 'r') as f:
    all_ingredients = json.load(f)

df_ingredients = pd.DataFrame(all_ingredients)
DisplayTable(df_ingredients.sample(100), 'Spoonacular Ingredients')


### Download the ingredient images

In [ ]:
IMAGE_DIR = "pantry_images"
SIZE = "100x100"
os.makedirs(IMAGE_DIR, exist_ok=True)

for item in all_ingredients:
    # 1. Construct the URL
    img_name = item['image']  # e.g., 'apple.jpg'
    img_url = f"https://img.spoonacular.com/ingredients_{SIZE}/{img_name}"
    
    # 2. Define local save path
    save_path = os.path.join(IMAGE_DIR, img_name)
    
    # 3. Download and Save
    try:
        response = requests.get(img_url, stream=True)
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(response.content)
            print(f"Downloaded: {img_name}")
        else:
            print(f"Failed: {img_name} (Status: {response.status_code})")
    except Exception as e:
        print(f"Error downloading {img_name}: {e}")